**Using GPT-2 to generate text:**
- as u might expect, we can use the transformers library to download GPT-2. By default, we get the small version (124M parameters)

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [44]:
model_id = "gpt2-xl"
# load GPT-2's pretrained tokenizer and the model itself
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

- to load the model, we use AutoModelForCausalLM.from_pretrained() => returns an instance of the appropriate class based on the checkpoint we ask for (in this case GPT2LMHeadModel). Since its a causal language model, its capable of generating text. Including the causal lm head implies adding a linear layer head on top of the decoder only model than transforms the contextual embeddings from shape (B, seq_len, embed_dim) to (B, seq_len, vocab_size). 
- device_map => auto option tells the function to automatically place the model on the best available device, typically the GPU. 
- dtype => auto: asks the function to choose the most appropriate data type for the model weights, based on whats available in the model checkpoint and available hardware. typically loads the model using 16-bit floats if hardware supports it or 32-bit floats. using half precision (16-bit) uses half the memory, letting you load larger models, and also gives the model a substantial boost

In [74]:
inz = gpt2_tokenizer("Hello my name is", return_tensors="pt").to(gpt2.device)
outz = gpt2.generate(**inz, max_new_tokens=20, pad_token_id=gpt2_tokenizer.eos_token_id)

In [75]:
outz

tensor([[15496,   616,  1438,   318,  1757, 31780,    13,   314,   716,   257,
           582,    13,   314,   716,   257,   582,   508,   468,   587, 16110,
            13,   314,   716,   257]], device='mps:0')

In [45]:
# wrapper around the model's generate() method to make it easy to generate text

def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id = tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

decoder only models often pad on the left side, for more efficient generation, since new tokens are added on the right..

In [46]:
prompt = "Scientists found a talking unicorn today. Here's the full story:"
generate(gpt2,gpt2_tokenizer,prompt)

'Scientists found a talking unicorn today. Here\'s the full story:\n\nA team of scientists from the University of California, Berkeley, has discovered a new species of unicorn, a creature that has been extinct for more than a century.\n\nThe new species, named the "unicorn-like" or "'

starts well then repeats itself alot. here's whats happening. by default, the generate() method simply picks the most likely token at eacg step, which is fine when u expect a very structured output, or for tasks such as question answering. but for creative writing it often gets the model stuck in a loop producing repetitive and uninteresting text. to fix, we set do_sample=True to make the egenerate method randomly sample each token based on the model's estimated probabilities for the possible tokens

In [47]:
import torch

In [49]:
torch.manual_seed(42)
prompt = "I've got a good feeling about Arsenal this season."
print(generate(gpt2, gpt2_tokenizer, prompt, do_sample=True))

I've got a good feeling about Arsenal this season. I really am.

"The game plan was exactly the same as what we did in the last season's cup final so I was confident we'd win it again."

And when Wenger said the following week, with just one game to


To get better results - play with the generate method's arguments such as:
- temperature:
  - defaults to 1: decrease for more predictable outcomes or increase for more diversity
- top_k:
  - only sample from the top k most probable tokens
- top_p:
 - restricting sampling to the smallest set of most probable tokens whose top probability is at least top_p
- num_beams
  - the beam width for beam search => defaults to 1: no beam search

- Top-p sampling (aka nucleus sampling) - often preferred over top-k sampling as it adaapts to the probability distribution

In [51]:
torch.manual_seed(42)

print(generate(gpt2, gpt2_tokenizer, prompt, do_sample=True, top_p=0.8))

I've got a good feeling about Arsenal this season. I really do.

"The players have worked hard, and that's a huge factor. It is what it is – I've got a great feeling about this team."


**Using GPT-2 For Question Answering:**
- writing a little function that takes a country name and asks gpt-2 to return its capital city:

In [52]:
DEFAULT_TEMPLATE = "Capital city of France = Paris \nCapital city of {country} = "

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

starts by creating a prompt from a prompt template: replaces the {country} placeholder with a given country name. Note that the prompt template includes an example of the task to help GPT-2 understand what to do and what format we expect. thats in-context learning. The function the calls generate fn to add 10 tokens to the prompt. more than we need to write the capital. lastly we do a bit opf post-processing by removing the initial prompt as well as anything after the first line

In [58]:
get_capital_city(gpt2, gpt2_tokenizer, "Scotland")

'Edinburgh'

In [59]:
get_capital_city(gpt2, gpt2_tokenizer, "UK")

'London'

In [60]:
get_capital_city(gpt2, gpt2_tokenizer, "Uganda")

'Kampala \xa0(formerly known as'

In [63]:
get_capital_city(gpt2, gpt2_tokenizer, "Canada")

'Ottawa'

base gpt-2 wasn't doing so well... so.. went for gpt2-xl

**testing out even larger models....**

**Downloading and Running an Even Larger Model:**
- **Mistral-7B**
   - Mistral-7B is a decoder-only model released by french startup Mistral AI. As suggested by its name - its got 7Billion params and implements several advanced transformers techniques such as grouped-query attention and sliding window to increase speed and performance. 

In [68]:
from huggingface_hub import login

In [70]:
checkpoint = "mistralai/Mistral-7B-v0.3"

In [72]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint, cache_dir="./mistral/")

ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.